In [1]:
import os
os.chdir('..')

In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import models
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch.multiprocessing as mp
from src.data.cwru_data import cwru_transform
from src.features.cwt import cwt2scalogram
import timm
import onnxruntime as ort

d:\anaconda3\envs\pytorchgpu\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
class MobileOneS0(nn.Module):
    def __init__(self, num_classes=4):
        super().__init__()
        self.model = timm.create_model("mobileone_s0")

        self.model.stem.conv_kxk[0].conv = nn.Conv2d(1, 48, (3, 3), (1, 1), (1, 1), bias=False)
        self.model.stem.conv_scale.conv = nn.Conv2d(1, 48, (1, 1), (1, 1), bias=False)

        self.model.head.fc = nn.Linear(1024, num_classes, bias=True)

    def forward(self, x):
        return self.model(x)

In [8]:
class EdgeNetXXS(nn.Module):
    def __init__(self, num_classes=4):
        super().__init__()
        self.model = timm.create_model('edgenext_xx_small.in1k', pretrained=False)

        self.model.stem[0] = nn.Conv2d(1, 24, (2, 2), (2, 2))
        
        self.model.head.fc = nn.Linear(168, num_classes, bias=True)

    def forward(self, x):
        return self.model(x)

In [10]:
class MobileNetV4(nn.Module):
    def __init__(self, num_classes=4):
        super().__init__()
        self.model = timm.create_model("mobilenetv4_conv_small")

        self.model.conv_stem = nn.Conv2d(1, 32, (3, 3), (1, 1), (1, 1))
        
        self.model.classifier = nn.Linear(1280, num_classes, bias=True)

    def forward(self, x):
        return self.model(x)

In [12]:
class GhostNetV2(nn.Module):
    def __init__(self, num_classes=4):
        super().__init__()
        self.model = timm.create_model("ghostnetv2_100")

        self.model.conv_stem = nn.Conv2d(1, 16, (3, 3), (1, 1), (1, 1))

        self.model.classifier = nn.Linear(1280, 4, bias=True)

    def forward(self, x):
        return self.model(x)

In [4]:
def export_onnx(model_class, num_classes, weight_path, onnx_path):
    state_dict = torch.load(weight_path, map_location='cpu')
    # fixed_state_dict = {k.replace('module.', 'model.', 1): v for k, v in state_dict.items()}

    paradigm = model_class(num_classes)
    paradigm.load_state_dict(state_dict)
    paradigm.eval()

    dummy = torch.randn(1, 1, 128, 128)
    torch.onnx.export(paradigm, dummy, onnx_path,
                      input_names=['input'], output_names=['output'],
                      dynamic_axes={'input': {0: 'batch'}, 'output': {0: 'batch'}},
                      opset_version=17)
    print(f"Exported → {onnx_path}")

In [20]:
root_path = r"D:\Capstone\models\ONNX\CNN\SPECTROGRAM\GhostNetV2"

weight_path = r"D:\Capstone\models\PT\CNN\SPECTROGRAM\GhostNetV2\best_model.pt"
save_path = weight_path.split('\\')[-1].split('.')[0] + '.onnx'
onnx_path = os.path.join(root_path, save_path)

export_onnx(GhostNetV2, 4, weight_path, onnx_path)

C:\Users\VUONG\AppData\Local\Temp\ipykernel_20060\3680747794.py:2: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state_dict = torch.load(weight_path, map_location='cpu')


Exported → D:\Capstone\models\ONNX\CNN\SPECTROGRAM\GhostNetV2\best_model.onnx
